# System Tour: Data Ingestion & Backtesting

Interactive walkthrough of the two-layer data lake, trade simulator, and walk-forward backtest harness.

**Sections**
1. [Lake structure](#1-lake-structure)
2. [Tiingo equity data](#2-tiingo-equity-data)
3. [FRED macro data](#3-fred-macro-data)
4. [Walk-forward splits](#4-walk-forward-splits)
5. [Trade simulator](#5-trade-simulator)
6. [Performance metrics](#6-performance-metrics)
7. [Full walk-forward backtest](#7-full-walk-forward-backtest)
8. [Backtest report](#8-backtest-report)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.patches import Patch

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

print('Environment ready.')

In [ ]:
import quant.storage.lake as lake
import quant.storage.catalog as catalog
from quant.config import settings
from quant.backtest.walkforward import walkforward_splits
from quant.backtest.simulator import simulate
from quant.backtest.metrics import compute_metrics
from quant.backtest.harness import run_backtest
from quant.backtest.report import format_report, summary_table

print('Data root :', settings.data_root)
print('Raw dir   :', settings.raw_dir)
print('Processed :', settings.processed_dir)

---
## 1 — Lake structure

Two layers:
- **raw/** — immutable per-day Parquet drops, one file per ingestion date
- **processed/** — hive-partitioned by `year=YYYY/month=MM`, rebuilt on each ingest run

In [ ]:
def lake_stats() -> pd.DataFrame:
    rows = []
    for layer_name, layer_path in [('raw', settings.raw_dir), ('processed', settings.processed_dir)]:
        if not layer_path.exists():
            continue
        for dataset in sorted(layer_path.iterdir()):
            if not dataset.is_dir():
                continue
            files = list(dataset.rglob('*.parquet'))
            size_mb = sum(f.stat().st_size for f in files) / 1_048_576
            rows.append({'layer': layer_name, 'dataset': dataset.name,
                         'files': len(files), 'size_mb': round(size_mb, 2)})
    return pd.DataFrame(rows)

stats = lake_stats()
print(stats.to_string(index=False) if not stats.empty else 'Lake is empty — run an ingest flow first.')

In [ ]:
if not stats.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(stats) * 0.4)))
    colors = {'raw': '#4C72B0', 'processed': '#DD8452'}
    for _, row in stats.iterrows():
        ax.barh(f"{row['layer']}/{row['dataset']}", row['size_mb'],
                color=colors.get(row['layer'], 'gray'))
    ax.set_xlabel('Size (MB)')
    ax.set_title('Data lake — file sizes by dataset')
    ax.legend(handles=[Patch(color=c, label=l) for l, c in colors.items()], loc='lower right')
    plt.tight_layout()
    plt.show()

---
## 2 — Tiingo equity data

Queried via DuckDB directly from the hive-partitioned Parquet files.

In [ ]:
try:
    eq = catalog.query(f"""
        SELECT symbol, timestamp, open, high, low, close, volume,
               adjClose, adjOpen, adjVolume, divCash, splitFactor
        FROM {catalog.table('equity_eod_tiingo')}
        ORDER BY symbol, timestamp
    """)
    eq['timestamp'] = pd.to_datetime(eq['timestamp'])
    eq = eq.set_index('timestamp')
    symbols = sorted(eq['symbol'].unique())
    print(f"{len(eq):,} rows | {eq.index.min().date()} to {eq.index.max().date()}")
    print(f"Symbols ({len(symbols)}): {symbols}")
    display(eq.head(10))
except Exception as e:
    print(f'No Tiingo data yet: {e}')
    eq = pd.DataFrame()
    symbols = []

In [ ]:
if not eq.empty:
    eq_m = eq.copy()
    eq_m['ym'] = eq_m.index.to_period('M')
    coverage = eq_m.groupby(['symbol', 'ym']).size().unstack('symbol').notna().astype(int)

    fig, ax = plt.subplots(figsize=(max(6, len(symbols) * 1.2), 5))
    ax.imshow(coverage.T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
    ax.set_yticks(range(len(symbols)))
    ax.set_yticklabels(symbols)
    step = max(1, len(coverage) // 12)
    ax.set_xticks(range(0, len(coverage), step))
    ax.set_xticklabels([str(p) for p in coverage.index[::step]], rotation=45, ha='right')
    ax.set_title('Tiingo coverage heatmap (blue = data present)')
    plt.tight_layout()
    plt.show()

In [ ]:
if not eq.empty:
    sym = symbols[0]
    s = eq[eq['symbol'] == sym].copy()

    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

    axes[0].plot(s.index, s['adjClose'], linewidth=1)
    axes[0].set_ylabel('Adj Close ($)')
    axes[0].set_title(f'{sym} — price, volume, corporate actions')

    axes[1].bar(s.index, s['adjVolume'], width=1, alpha=0.6)
    axes[1].set_ylabel('Adj Volume')
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

    divs = s[s['divCash'] > 0]
    splits = s[s['splitFactor'] != 1.0]
    axes[2].bar(divs.index, divs['divCash'], width=5, color='steelblue', label='divCash')
    axes[2].set_ylabel('divCash ($)', color='steelblue')
    ax_r = axes[2].twinx()
    ax_r.scatter(splits.index, splits['splitFactor'], color='darkorange', zorder=5, label='splitFactor')
    ax_r.set_ylabel('splitFactor', color='darkorange')
    axes[2].set_xlabel('Date')
    h1, l1 = axes[2].get_legend_handles_labels()
    h2, l2 = ax_r.get_legend_handles_labels()
    axes[2].legend(h1 + h2, l1 + l2, loc='upper left', fontsize=9)

    plt.tight_layout()
    plt.show()

---
## 3 — FRED macro data

In [ ]:
try:
    fred = catalog.query(f"""
        SELECT series_id, timestamp, value
        FROM {catalog.table('macro_fred')}
        ORDER BY series_id, timestamp
    """)
    fred['timestamp'] = pd.to_datetime(fred['timestamp'])
    series_list = sorted(fred['series_id'].unique())
    print(f"{len(fred):,} rows | {len(series_list)} series: {series_list}")
    display(fred.head(10))
except Exception as e:
    print(f'No FRED data yet: {e}')
    fred = pd.DataFrame()
    series_list = []

In [ ]:
if not fred.empty:
    n = len(series_list)
    cols = min(2, n)
    rows_count = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows_count, cols, figsize=(12, 3 * rows_count), squeeze=False)
    for idx, sid in enumerate(series_list):
        ax = axes[idx // cols][idx % cols]
        s = fred[fred['series_id'] == sid].set_index('timestamp')['value']
        ax.plot(s.index, s.values, linewidth=1)
        ax.set_title(sid)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    for idx in range(n, rows_count * cols):
        axes[idx // cols][idx % cols].set_visible(False)
    fig.suptitle('FRED macro series', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

---
## 4 — Walk-forward splits

The `walkforward_splits` generator implements **purged** rolling CV:
- **Purge**: remove training samples whose label window overlaps the test period
- **Embargo**: additional buffer to defeat residual serial correlation

In [ ]:
N, TRAIN, TEST, STEP, HORIZON, EMBARGO = 200, 100, 20, 20, 5, 3

splits = list(walkforward_splits(
    n_samples=N, train_window=TRAIN, test_window=TEST,
    step=STEP, label_horizon=HORIZON, embargo=EMBARGO,
))
print(f"{len(splits)} folds | train={TRAIN}, test={TEST}, step={STEP}, "
      f"horizon={HORIZON}, embargo={EMBARGO}")
for i, (tr, te) in enumerate(splits[:3]):
    print(f"  fold {i}: train [{tr[0]}..{tr[-1]}] ({len(tr)} bars)  "
          f"test [{te[0]}..{te[-1]}] ({len(te)} bars)")
if len(splits) > 3:
    print('  ...')

In [ ]:
fig, ax = plt.subplots(figsize=(12, max(3, len(splits) * 0.4)))

for fold_idx, (tr, te) in enumerate(splits):
    y = fold_idx
    nominal_start = te[0] - TRAIN
    ax.barh(y, TRAIN, left=nominal_start, height=0.6, color='#cccccc', alpha=0.5)
    if len(tr) > 0:
        ax.barh(y, len(tr), left=tr[0], height=0.6, color='#4C72B0', alpha=0.8)
    purge_start = te[0] - HORIZON - EMBARGO
    ax.barh(y, HORIZON + EMBARGO, left=purge_start, height=0.6, color='#C44E52', alpha=0.6)
    ax.barh(y, len(te), left=te[0], height=0.6, color='#55A868', alpha=0.8)

ax.set_yticks(range(len(splits)))
ax.set_yticklabels([f'Fold {i}' for i in range(len(splits))])
ax.set_xlabel('Sample index')
ax.set_title('Purged walk-forward splits')
ax.legend(handles=[
    Patch(color='#cccccc', alpha=0.5, label='Nominal train window'),
    Patch(color='#4C72B0', alpha=0.8, label='Effective train (post purge+embargo)'),
    Patch(color='#C44E52', alpha=0.6, label='Purge + embargo gap'),
    Patch(color='#55A868', alpha=0.8, label='Test window'),
], loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

---
## 5 — Trade simulator

MA(20) crossover on real Tiingo data — signal at bar t fills at bar t+1 open.

In [ ]:
if not eq.empty:
    sym = symbols[0]
    prices = eq[eq['symbol'] == sym][['open', 'high', 'low', 'close', 'volume']].copy()
    prices = prices.sort_index().dropna()

    ma20 = prices['close'].rolling(20).mean()
    raw_sig = np.where(prices['close'] > ma20, 1, -1)
    signals = pd.Series(raw_sig, index=prices.index, dtype=int)
    signals.iloc[:20] = 0  # flat during warmup

    equity_ma, trade_log_ma = simulate(
        prices=prices, signals=signals,
        initial_capital=100_000,
        commission_per_share=0.005,
        slippage_bps=5.0,
        liquidity_cap=0.10,
    )
    print(f'Trades: {len(trade_log_ma)} | Final equity: ${equity_ma.iloc[-1]:,.0f}')
    display(trade_log_ma.tail(5))

In [ ]:
if not eq.empty:
    fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

    axes[0].plot(prices.index, prices['close'], linewidth=1, label='Close')
    axes[0].plot(prices.index, ma20, linewidth=1, linestyle='--', label='MA(20)')
    if not trade_log_ma.empty:
        axes[0].scatter(trade_log_ma['date'], trade_log_ma['entry_price'],
                        marker='^', color='green', s=40, zorder=5, label='Entry')
        axes[0].scatter(trade_log_ma['date'], trade_log_ma['exit_price'],
                        marker='v', color='red', s=40, zorder=5, label='Exit')
    axes[0].set_ylabel('Price ($)')
    axes[0].set_title(f'{sym} — MA(20) crossover strategy')
    axes[0].legend(fontsize=9)

    axes[1].plot(equity_ma.index, equity_ma.values / 1000, linewidth=1, color='#4C72B0')
    axes[1].axhline(100, color='black', linewidth=0.5, linestyle='--')
    axes[1].set_ylabel('Portfolio ($k)')

    peak = equity_ma.cummax()
    dd = (equity_ma - peak) / peak * 100
    axes[2].fill_between(dd.index, dd.values, 0, color='#C44E52', alpha=0.4)
    axes[2].set_ylabel('Drawdown (%)')
    axes[2].set_xlabel('Date')

    plt.tight_layout()
    plt.show()

---
## 6 — Performance metrics

In [ ]:
if not eq.empty:
    returns_ma = equity_ma.pct_change().dropna()
    m = compute_metrics(returns_ma, trade_log=trade_log_ma if not trade_log_ma.empty else None)

    print(f'=== {sym} MA(20) metrics ===')
    for k, v in m.items():
        if k in ('total_return', 'annualized_return', 'max_drawdown', 'hit_rate'):
            print(f'  {k:<22} {v:.2%}')
        elif k == 'profit_factor':
            print(f'  {k:<22} {v:.3f}' if np.isfinite(v) else f'  {k:<22} {v}')
        else:
            print(f'  {k:<22} {v:.3f}')

In [ ]:
if not eq.empty:
    import matplotlib.colors as mcolors

    monthly = returns_ma.resample('ME').apply(lambda r: (1 + r).prod() - 1)
    mdf = monthly.to_frame('ret')
    mdf['year'] = mdf.index.year
    mdf['month'] = mdf.index.month
    pivot = mdf.pivot(index='year', columns='month', values='ret')
    pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun',
                     'Jul','Aug','Sep','Oct','Nov','Dec']

    vals = pivot.values[~np.isnan(pivot.values)]
    norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=vals.min(), vmax=vals.max())

    fig, ax = plt.subplots(figsize=(14, max(3, len(pivot) * 0.55)))
    im = ax.imshow(pivot.values, cmap='RdYlGn', norm=norm, aspect='auto')
    ax.set_xticks(range(12))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index)
    for r in range(len(pivot)):
        for c in range(12):
            v = pivot.values[r, c]
            if not np.isnan(v):
                ax.text(c, r, f'{v:.1%}', ha='center', va='center', fontsize=8)
    plt.colorbar(im, ax=ax, format=mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.set_title(f'{sym} MA(20) — monthly returns heatmap')
    plt.tight_layout()
    plt.show()

---
## 7 — Full walk-forward backtest

`run_backtest` with a `LogisticRegression` model and lagged log-return features.

**Label**: sign of the bar t+1 to t+2 return — because signal at t fills at t+1 open and earns the t+1 to t+2 move.

In [ ]:
if not eq.empty:
    from sklearn.linear_model import LogisticRegression

    sym = symbols[0]
    prices_bt = eq[eq['symbol'] == sym][['open', 'high', 'low', 'close', 'volume']].copy()
    prices_bt = prices_bt.sort_index().dropna()

    lags = [1, 2, 3, 5, 10]
    log_ret = np.log(prices_bt['close'] / prices_bt['close'].shift(1))
    features = pd.DataFrame(
        {f'lag_{l}': log_ret.shift(l) for l in lags},
        index=prices_bt.index,
    ).dropna()
    prices_bt = prices_bt.loc[features.index]

    close = prices_bt['close']
    fwd_ret = np.log(close.shift(-2) / close.shift(-1))
    labels = np.sign(fwd_ret)

    valid = features.index.intersection(labels.dropna().index)
    features = features.loc[valid]
    prices_bt = prices_bt.loc[valid]
    labels = labels.loc[valid]

    model = LogisticRegression(C=1.0, max_iter=500, random_state=42)

    print(f'Running backtest on {len(features)} bars of {sym}...')
    result = run_backtest(
        model=model,
        features=features,
        labels=labels,
        prices=prices_bt,
        train_window=252,
        test_window=63,
        step=63,
        label_horizon=2,
        embargo=5,
        initial_capital=100_000,
        commission_per_share=0.005,
        slippage_bps=5.0,
    )
    print(f'Done — {len(result.fold_metrics)} folds, {len(result.trade_log)} trades')

In [ ]:
if not eq.empty and 'result' in dir():
    ec = result.equity_curve
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

    axes[0].plot(ec.index, ec.values / 1000, linewidth=1, color='#4C72B0', label='OOS equity')
    axes[0].axhline(100, color='black', linewidth=0.5, linestyle='--')
    axes[0].set_ylabel('Portfolio ($k)')
    axes[0].set_title(f'{sym} — LogisticRegression walk-forward backtest (OOS equity)')
    axes[0].legend()

    peak = ec.cummax()
    dd = (ec - peak) / peak * 100
    axes[1].fill_between(dd.index, dd.values, 0, color='#C44E52', alpha=0.4)
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].set_xlabel('Date')

    plt.tight_layout()
    plt.show()

In [ ]:
if not eq.empty and 'result' in dir():
    fold_sharpes = [f.get('sharpe', float('nan')) for f in result.fold_metrics]
    fig, ax = plt.subplots(figsize=(max(6, len(fold_sharpes) * 0.7), 3))
    colors_bar = ['#55A868' if s > 0 else '#C44E52' for s in fold_sharpes]
    ax.bar(range(len(fold_sharpes)), fold_sharpes, color=colors_bar)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Fold')
    ax.set_ylabel('OOS Sharpe')
    ax.set_title('OOS Sharpe by fold')
    plt.tight_layout()
    plt.show()

---
## 8 — Backtest report

In [ ]:
if not eq.empty and 'result' in dir():
    print(format_report(result))

In [ ]:
if not eq.empty and 'result' in dir():
    tbl = summary_table(result)
    display(tbl.style.format(
        lambda v: '—' if (isinstance(v, float) and np.isnan(v)) else
                  f'{v:.2%}' if abs(v) <= 5 else f'{v:.3f}'
    ).background_gradient(cmap='RdYlGn', axis=None))

In [ ]:
if not eq.empty and 'result' in dir():
    oos = result.oos_metrics
    is_ = result.is_metrics
    keys = ['sharpe', 'total_return', 'hit_rate', 'max_drawdown']
    print('IS/OOS gap (positive = IS beats OOS = potential overfit):')
    for k in keys:
        gap = is_.get(k, float('nan')) - oos.get(k, float('nan'))
        warn = '  <<< WARN' if k == 'sharpe' and abs(gap) > 0.5 else ''
        print(f'  {k:<22} {gap:+.3f}{warn}')